# Day 12

Data Source:
- Global Nuclear Power Tracker (Global Energy Monitor, August 2026): https://globalenergymonitor.org/projects/global-nuclear-power-tracker#download
- Yearly Electricity Data (Ember): https://ember-energy.org/data/yearly-electricity-data/
- Admin 0 Countries, 1:50m (Natural Earth): https://www.naturalearthdata.com/downloads/50m-cultural-vectors/50m-admin-0-countries-2/

### 1. Import libraries and packages

In [60]:
import os
import urllib.request

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

### Configurations
Files, base year, colours, text and inset areas.

In [62]:
# data source
GEM_XLSX = "Global-Nuclear-Power-Tracker-August-2026.xlsx"
EMBER_CSV = "release_generation_yearly_global.csv"
NE_FILE = "ne_50m_admin_0_countries.geojson"
NE_URL = ("https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/"
          "geojson/ne_50m_admin_0_countries.geojson")

In [63]:
# outputs
STATS_CSV = "nuclear_change_by_country.csv"
POSTER_PNG = "nuclear_poster.png"

In [64]:
BASE_YEAR = 2010        # capacity at the end of this year is the starting point (the year before Fukushima (2011)
NOW_LABEL = "Aug 2026"  # the tracker release
MIN_BAR_GW = 1.0        # countries with a smaller net change are left out of the bar chart

In [65]:
# GEM country names that don't match Ember / Natural Earth
# manual override required
ISO_OVERRIDES = {
    "Czech Republic": "CZE", "Taiwan": "TWN", "Türkiye": "TUR", "United States": "USA",
    "Philippines": "PHL", "Vietnam": "VNM",
}

# Lines in the share-of-electricity chart (Ember area names) and their colours.
LINE_COUNTRIES = {
    "France": "#2e6fb7",
    "Japan": "#d1452e",
    "Germany": "#8a5a9e",
    "China": "#d99a1c",
    "United Arab Emirates": "#1f9e89",
}
WORLD_COLOUR = "#8c8579"
LINE_LABELS = {"United Arab Emirates": "UAE"}
LINE_START = 2000

# Zoom insets for crowded regions: (label, lon/lat box, where to put it on the map in
# Equal Earth metres: xmin, ymin, xmax, ymax, label corner). Set INSETS = [] to switch off.
# Put each label in a corner that falls on open sea, so it needs no background.
INSETS = [
    ("Europe",    (-11, 35, 33, 63),  (-17.2e6, -7.9e6, -8.6e6, -0.9e6), "top left"),
    ("East Asia", (100, 18, 146, 46), (4.9e6, -7.9e6, 10.4e6, -2.5e6), "bottom right"),
]

# Construction circles: area grows with the capacity being built at each site.
SIZE_KEY_MW = [500, 2000, 5000]   # example sizes shown in the legend

# Map classes for net change in GW: (upper limit, colour, legend label)
CLASSES = [
    (-10,    "#9c3a22", "Lost 10+ GW"),
    (-2,     "#d0704a", "Lost 2–10 GW"),
    (-0.0005, "#efb595", "Lost under 2 GW"),
    (0.0005,  "#cfc8ba", "No change"),      # same capacity as in 2010 (within 0.5 MW)
    (2,       "#9fd3c7", "Gained under 2 GW"),
    (10,     "#45a594", "Gained 2–10 GW"),
    (np.inf, "#1d6b5f", "Gained 10+ GW"),
]
NO_NUCLEAR = "#ece6da"

BG = "#f5f1e8"
INK = "#2b2b2b"
INK_SOFT = "#666666"
GRID = "#e2ddd2"

TITLE = "Who's going nuclear and who's leaving?"
SUBTITLE = (f"Change in operating nuclear capacity since the end of {BASE_YEAR}, "
            "the year before Fukushima")
CREDIT = ("Reactors: Global Energy Monitor, Global Nuclear Power Tracker, August 2026 (CC BY 4.0)  ·  "
          "Electricity: Ember, Yearly Electricity Data")
POSTER_SIZE = (14, 18)  # inches, saved at 300 dpi

## Data
Loads the GEM tracker, Ember and Natural Earth, and builds the country table.

In [67]:
def load_gem():# read Global Energy Monitor dataset/Global Nuclear Power Tracker
    return pd.read_excel(GEM_XLSX, sheet_name="Data")

def load_ember(): # read Ember dataset/Share of Electricity Generation
    return pd.read_csv(EMBER_CSV)

def load_world(): # load Natural Earth dataset
    if not os.path.exists(NE_FILE):
        print("Downloading Natural Earth country outlines ...")
        urllib.request.urlretrieve(NE_URL, NE_FILE)
    world = gpd.read_file(NE_FILE)
    world = world[world["ADM0_A3"] != "ATA"]  # drop Antarctica
    return world[["ADM0_A3", "ADMIN", "geometry"]].to_crs("EPSG:8857")  # Equal Earth

# builds a table from country name to map code using Ember's list plus overrides
def iso_lookup(ember):
    countries = ember[ember["Area type"] == "Country or economy"]
    lookup = dict(zip(countries["Area"], countries["ISO 3 code"]))
    lookup.update(ISO_OVERRIDES)
    return lookup


def country_table(gem, lookup):
    """Capacity at the end of BASE_YEAR vs now, by country, in GW."""
    # reactors that have actually run at some point of time
    built = gem["Status"].isin(["operating", "retired", "mothballed"]) & gem["Start Year"].notna()
    # do not count reactors that started and shut down by 2010
    on_base = (built & (gem["Start Year"] <= BASE_YEAR)
               & (gem["Retirement Year"].isna() | (gem["Retirement Year"] > BASE_YEAR)))

    def gw(mask):
        return gem[mask].groupby("Country/Area")["Capacity (MW)"].sum() / 1000

    t = pd.DataFrame({
        f"Capacity end {BASE_YEAR} (GW)": gw(on_base),
        "Operating now (GW)": gw(gem["Status"] == "operating"),
        f"Added since {BASE_YEAR} (GW)": gw(built & (gem["Start Year"] > BASE_YEAR)),
        f"Retired since {BASE_YEAR} (GW)": gw((gem["Status"] == "retired")
                                             & (gem["Retirement Year"] > BASE_YEAR)),
        "Mothballed (GW)": gw((gem["Status"] == "mothballed")
                              & (gem["Start Year"] <= BASE_YEAR)),
        "Under construction (GW)": gw(gem["Status"] == "construction"),
    }).fillna(0)
    t["Net change (GW)"] = t["Operating now (GW)"] - t[f"Capacity end {BASE_YEAR} (GW)"]
    # Keep countries that had reactors in 2010, have them now, or are building them
    keep = t[[f"Capacity end {BASE_YEAR} (GW)", "Operating now (GW)",
              "Under construction (GW)"]].sum(axis=1) > 0
    t = t[keep].copy()
    t.index.name = "Country"
    t["ISO3"] = [lookup.get(c) for c in t.index]
    missing = t[t["ISO3"].isna()].index.tolist()
    if missing:
        print(f"No ISO code for {missing}; add them to ISO_OVERRIDES to show them on the map.")

    base = t[f"Capacity end {BASE_YEAR} (GW)"]
    now = t["Operating now (GW)"]
    t["Group"] = "existing"
    t.loc[(base > 0) & (now == 0), "Group"] = "left" # had capacity in 2010, no capacity now
    t.loc[(base == 0) & (now > 0), "Group"] = "newcomer" # no capacity in2010, some now
    t.loc[(base == 0) & (now == 0) & (t["Under construction (GW)"] > 0), "Group"] = "first build" #none then or now but building
    return t.sort_values("Net change (GW)")

# group reactors under construction by plant, adds up their capacity and takes the average location
def construction_sites(gem):
    c = gem[(gem["Status"] == "construction") & gem["Latitude"].notna()]
    sites = (c.groupby(["Country/Area", "Project Name"])
              .agg(MW=("Capacity (MW)", "sum"), lat=("Latitude", "mean"),
                   lon=("Longitude", "mean"))
              .reset_index())
    pts = gpd.GeoDataFrame(sites, geometry=gpd.points_from_xy(sites.lon, sites.lat),
                           crs="EPSG:4326")
    return pts.to_crs("EPSG:8857")

# goes through CLASSES in order and returns the first colour whose upper limit the net change doesn’t exceed.
def class_colour(net):
    for upper, colour, _ in CLASSES:
        if net <= upper:
            return colour
    return CLASSES[-1][1]

## Map
Choropleth of net change, construction sites, and zoom insets.

In [69]:
# draws 1. countries filled by colour; 2. hatching on countries that left nuclear;
# 3. a thick outline on newcoemrs; 4. hollow construction circles
def plot_layers(ax, w, left, new, sites, scale=1.0):
    w.plot(ax=ax, color=w["fill"], edgecolor="white", linewidth=0.35 * scale)
    left.plot(ax=ax, facecolor="none", edgecolor=INK, hatch="////", linewidth=0.6)
    new.plot(ax=ax, facecolor="none", edgecolor=INK, linewidth=1.4)
    sizes = circle_size(sites["MW"], scale)
    ax.scatter(sites.geometry.x, sites.geometry.y, s=sizes, facecolors="none",
               edgecolors=INK, linewidths=0.9, zorder=5)

# recolours country by net change
def draw_map(ax, world, table, sites):
    by_iso = table.dropna(subset=["ISO3"]).set_index("ISO3")
    w = world.copy()
    w["fill"] = NO_NUCLEAR
    for iso, row in by_iso.iterrows():
        sel = w["ADM0_A3"] == iso
        # countries building their first reactor have none yet, so they show as "No nuclear
        # power"; their construction circles show what's coming
        w.loc[sel, "fill"] = (NO_NUCLEAR if row["Group"] == "first build"
                              else class_colour(row["Net change (GW)"]))
    left = w[w["ADM0_A3"].isin(by_iso[by_iso["Group"] == "left"].index)]
    new = w[w["ADM0_A3"].isin(by_iso[by_iso["Group"] == "newcomer"].index)]

    plot_layers(ax, w, left, new, sites)
    ax.set_xlim(-17.3e6, 17.3e6)
    ax.set_ylim(-8.0e6, 8.5e6)
    ax.set_aspect("equal")
    ax.set_axis_off()

    # zoom insets: drawn in empty ocean, with a box marking the area on the main map
    for label, (lon0, lat0, lon1, lat1), (x0, y0, x1, y1), corner in INSETS:
        box = gpd.GeoSeries.from_xy([lon0, lon1], [lat0, lat1], crs="EPSG:4326").to_crs(
            "EPSG:8857")
        bx0, by0 = box.x.min(), box.y.min()
        bx1, by1 = box.x.max(), box.y.max()
        ins = ax.inset_axes([x0, y0, x1 - x0, y1 - y0], transform=ax.transData)
        plot_layers(ins, w, left, new, sites, scale=1.6)
        # match the inset's shape to its box so the zoom isn't stretched
        cx, cy = (bx0 + bx1) / 2, (by0 + by1) / 2
        aspect = (x1 - x0) / (y1 - y0)
        half_w = max((bx1 - bx0) / 2, (by1 - by0) / 2 * aspect)
        half_h = half_w / aspect
        ins.set_xlim(cx - half_w, cx + half_w)
        ins.set_ylim(cy - half_h, cy + half_h)
        ins.set_aspect("auto")
        ins.set_xticks([])
        ins.set_yticks([])
        ins.set_xlabel("")
        ins.set_ylabel("")
        ins.set_facecolor(BG)
        for sp in ins.spines.values():
            sp.set_edgecolor(INK)
            sp.set_linewidth(0.8)
        right, bottom = "right" in corner, "bottom" in corner
        ins.text(0.97 if right else 0.03, 0.04 if bottom else 0.96, label,
                 transform=ins.transAxes, fontsize=11, fontweight="bold", color=INK,
                 ha="right" if right else "left", va="bottom" if bottom else "top",
                 zorder=10)
        ax.add_patch(plt.Rectangle((cx - half_w, cy - half_h), 2 * half_w, 2 * half_h,
                                   fill=False, edgecolor=INK, linewidth=0.7,
                                   linestyle=(0, (3, 2)), zorder=6))

def map_legend(fig, y1, y2, y3, fs=1.0):
    patches = [Patch(facecolor=c, edgecolor="none", label=lab) for _, c, lab in CLASSES]
    leg1 = fig.legend(handles=patches, loc="center", bbox_to_anchor=(0.5, y1),
                      ncol=len(patches), frameon=False, fontsize=10.5 * fs,
                      handlelength=1.4, handleheight=1.0, columnspacing=1.4)
    fig.add_artist(leg1)
    extras = [
        Patch(facecolor=NO_NUCLEAR, edgecolor="none", label="No nuclear power"),
        Patch(facecolor="#d0704a", edgecolor=INK, hatch="////", label="Left nuclear since 2010"),
        Patch(facecolor="#45a594", edgecolor=INK, linewidth=1.4, label="New to nuclear since 2010"),
    ]
    sizes = [Line2D([0], [0], marker="o", color="none", markerfacecolor="none",
                    markeredgecolor=INK, markeredgewidth=0.9,
                    markersize=np.sqrt(circle_size(mw)),
                    label=f"{mw / 1000:g} GW") for mw in SIZE_KEY_MW]
    fig.text(0.5, y3, "Reactor sites under construction, sized by capacity being built:",
             ha="right", va="center", fontsize=10.5 * fs, color=INK)
    leg3 = fig.legend(handles=sizes, loc="center left", bbox_to_anchor=(0.505, y3),
                      ncol=len(sizes), frameon=False, fontsize=10.5 * fs,
                      handlelength=1.2, columnspacing=1.6, handletextpad=0.5)
    fig.add_artist(leg3)
    fig.legend(handles=extras, loc="center", bbox_to_anchor=(0.5, y2), ncol=len(extras),
               frameon=False, fontsize=10.5 * fs, handlelength=1.4, handleheight=1.0,
               columnspacing=1.4)

def circle_size(mw, scale=1.0):
    """Marker area (points²) for a site building `mw` megawatts."""
    return (12 + np.sqrt(mw) * 2.2) * scale

## Charts
Bar chart of net change and line chart of nuclear's share of electricity.

In [71]:
def draw_bars(ax, table, fs=1.0):
    t = table[table["Net change (GW)"].abs() >= MIN_BAR_GW].sort_values("Net change (GW)")
    y = np.arange(len(t))
    vals = t["Net change (GW)"].values
    ax.barh(y, vals, color=[class_colour(v) for v in vals], height=0.72, zorder=3)
    ax.axvline(0, color=INK, linewidth=0.9, zorder=4)
    ax.set_yticks(y, t.index, fontsize=10 * fs, color=INK)
    span = np.abs(vals).max()
    for yi, v in zip(y, vals):
        ax.text(v + (0.012 * span if v >= 0 else -0.012 * span), yi, f"{v:+.1f}",
                va="center", ha="left" if v >= 0 else "right", fontsize=9 * fs, color=INK)
    if "Japan" in t.index:
        j = t.loc["Japan"]
        yi = list(t.index).index("Japan")
        if j["Mothballed (GW)"] > 0:
            ax.text(j["Net change (GW)"] * 0.5, yi,
                    f"{j['Mothballed (GW)']:.0f} GW mothballed",
                    ha="center", va="center", fontsize=8.5 * fs, color="white",
                    style="italic", zorder=5)
    ax.set_xlim(vals.min() - 0.18 * span, vals.max() + 0.18 * span)
    ax.set_xlabel(f"Net change since {BASE_YEAR} (GW)", fontsize=10 * fs, color=INK_SOFT)
    ax.grid(axis="x", color=GRID, linewidth=0.8, zorder=0)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color("#999999")
    ax.tick_params(axis="y", length=0)
    ax.tick_params(axis="x", colors=INK_SOFT, labelsize=9.5 * fs)
    ax.set_facecolor(BG)


def draw_lines(ax, ember, fs=1.0):
    n = ember[(ember["Electricity source"] == "Nuclear") & (ember["Year"] >= LINE_START)]
    share = n.pivot_table(index="Year", columns="Area", values="Share of generation (%)")
    ends = []
    if "World" in share:
        s = share["World"].dropna()
        ax.plot(s.index, s.values, color=WORLD_COLOUR, linewidth=2.2, linestyle=(0, (4, 2)),
                zorder=3)
        ends.append([s.values[-1], f"World average {s.values[-1]:.0f}%", WORLD_COLOUR])
    for area, colour in LINE_COUNTRIES.items():
        if area not in share:
            print(f"'{area}' not found in the Ember file; skipping")
            continue
        s = share[area].dropna()
        ax.plot(s.index, s.values, color=colour, linewidth=2.4, zorder=4)
        ends.append([s.values[-1], f"{LINE_LABELS.get(area, area)} {s.values[-1]:.0f}%", colour])

    ends.sort()
    gap = 4.2
    ends[0][0] = max(ends[0][0], 1.5)
    for i in range(1, len(ends)):
        ends[i][0] = max(ends[i][0], ends[i - 1][0] + gap)
    last = share.index.max()
    for y_lab, text, colour in ends:
        ax.plot([last + 0.3, last + 0.9], [y_lab, y_lab], color=colour, linewidth=2.4,
                clip_on=False)
        ax.text(last + 1.2, y_lab, text, va="center", fontsize=9.5 * fs, color=INK,
                clip_on=False)

    ax.axvline(2011, color=INK_SOFT, linewidth=0.8, linestyle=":", zorder=2)
    ax.text(2011.3, 86, "Fukushima\n2011", fontsize=8.5 * fs, color=INK_SOFT, va="top")
    ax.set_xlim(LINE_START, last)
    ax.set_ylim(0, 90)
    ax.set_ylabel("% of the country's electricity from nuclear", fontsize=10 * fs,
                  color=INK_SOFT)
    ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color("#999999")
    ax.tick_params(axis="y", length=0, colors=INK_SOFT, labelsize=9.5 * fs)
    ax.tick_params(axis="x", colors=INK_SOFT, labelsize=9.5 * fs)
    ax.set_facecolor(BG)

## Poster
Puts everything together.

In [73]:
def save_figure(fig, path, dpi):
    try:
        fig.savefig(path, dpi=dpi, facecolor=BG)
        return path
    except OSError as err:
        from datetime import datetime
        stem, ext = os.path.splitext(path)
        alt = f"{stem}_{datetime.now():%H%M%S}{ext}"
        print(f"Couldn't overwrite {path} ({err}); saving as {alt}")
        fig.savefig(alt, dpi=dpi, facecolor=BG)
        return alt


def make_poster(gem, ember, world, table, sites):
    fs = 1.1
    fig = plt.figure(figsize=POSTER_SIZE, facecolor=BG)
    m = 0.045

    fig.text(m, 0.972, TITLE, fontsize=32, fontweight="bold", color=INK, va="center")
    fig.text(m, 0.950, SUBTITLE, fontsize=15, color=INK_SOFT, va="center")

    map_ax = fig.add_axes([0.02, 0.555, 0.96, 0.375])
    draw_map(map_ax, world, table, sites)
    map_legend(fig, 0.545, 0.522, 0.499, fs=0.95)

    fig.add_artist(Line2D([m, 1 - m], [0.480, 0.480], color="#c8c2b5", lw=1,
                          transform=fig.transFigure))

    fig.text(m, 0.455, "Change in operating nuclear capacity by country", fontsize=14,
             fontweight="bold", color=INK, va="center")
    bar_ax = fig.add_axes([0.16, 0.075, 0.33, 0.355])
    draw_bars(bar_ax, table, fs=fs)

    fig.text(0.56, 0.455, "Share of each country's own electricity from nuclear",
             fontsize=14, fontweight="bold", color=INK, va="center")
    line_ax = fig.add_axes([0.60, 0.075, 0.27, 0.34])
    draw_lines(line_ax, ember, fs=fs)

    fig.text(0.5, 0.015, CREDIT, ha="center", va="bottom", fontsize=9, color=INK_SOFT)
    out = save_figure(fig, POSTER_PNG, dpi=300)
    plt.close(fig)
    print(f"Saved {out}")


def main():
    gem = load_gem()
    ember = load_ember()
    world = load_world()
    table = country_table(gem, iso_lookup(ember))
    table.round(2).to_csv(STATS_CSV)
    print(f"Saved {STATS_CSV}")
    print(table[["Net change (GW)", "Group"]].round(1).to_string())
    sites = construction_sites(gem)
    make_poster(gem, ember, world, table, sites)

## Run

In [75]:
main()

Saved nuclear_change_by_country.csv
                      Net change (GW)        Group
Country                                           
Japan                           -28.3     existing
Germany                         -21.5         left
United States                    -7.0     existing
United Kingdom                   -5.4     existing
Taiwan                           -5.1         left
Sweden                           -3.0     existing
Belgium                          -2.1     existing
Canada                           -0.7     existing
Spain                            -0.5     existing
Switzerland                      -0.4     existing
France                           -0.2     existing
Hungary                           0.0     existing
Brazil                            0.0     existing
Bulgaria                          0.0     existing
Czech Republic                    0.0     existing
Egypt                             0.0  first build
South Africa                      0.0     exis